<a href="https://colab.research.google.com/github/lizbethQCC/2024/blob/main/Detecion_de_objetos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ultralytics opencv-python -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 10.6 MB/s eta 0:00:00


In [4]:
from google.colab import files

uploaded = files.upload()

video_path = next(iter(uploaded))
print("Video:", video_path)

Saving video_detectado.mp4 to video_detectado.mp4
Video: video_detectado.mp4


In [5]:
from ultralytics import YOLO
import cv2

# Cargar modelo YOLO
model = YOLO("yolo11n.pt")

# Abrir video
cap = cv2.VideoCapture(video_path)

# Propiedades del video
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Video de salida
output_path = "video_detectado.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Detectar objetos
    results = model(frame, verbose=False)

    # Contador de personas
    personas = 0

    for result in results:

        boxes = result.boxes

        for box in boxes:

            clase = int(box.cls[0])
            confianza = float(box.conf[0])

            # Clase 0 = persona en COCO
            if clase == 0 and confianza > 0.5:
                personas += 1

    # Dibujar detecciones
    annotated_frame = results[0].plot()

    # Mostrar contador
    cv2.putText(
        annotated_frame,
        f"Personas: {personas}",
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 255, 0),
        3
    )

    # Guardar frame
    out.write(annotated_frame)

cap.release()
out.release()

print("Video terminado:", output_path)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Video terminado: video_detectado.mp4


In [7]:
from google.colab import files

files.download("video_detectado.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from IPython.display import HTML
from base64 import b64encode
import os

# 1) Verificar que el video existe y es válido
archivo = "video_detectado.mp4"

if not os.path.exists(archivo):
    print("❌ No se encontró", archivo)
else:
    size_mb = os.path.getsize(archivo) / 1024 / 1024
    print(f"✅ Archivo encontrado: {size_mb:.2f} MB")

    # 2) Convertir a H.264 (el códec de OpenCV NO se reproduce en el navegador)
    print("Convirtiendo a formato compatible...")
    !ffmpeg -y -i video_detectado.mp4 -vcodec libx264 -preset fast -crf 23 -movflags +faststart video_final.mp4 -loglevel error

    # 3) Si pesa más de 20 MB, comprimir (Colab limita el HTML embebido)
    size_final = os.path.getsize("video_final.mp4") / 1024 / 1024
    if size_final > 20:
        print(f"Comprimiendo ({size_final:.1f} MB)...")
        !ffmpeg -y -i video_final.mp4 -vf "scale=640:-2" -c:v libx264 -crf 30 -an video_final.mp4 -loglevel error

    # 4) Mostrar el video
    data = b64encode(open("video_final.mp4", "rb").read()).decode()
    display(HTML(f'''
    <video width=800 controls>
        <source src="data:video/mp4;base64,{data}" type="video/mp4">
        Tu navegador no soporta video HTML5.
    </video>
    '''))
    print("✅ Video mostrado arriba")

✅ Archivo encontrado: 0.57 MB
Convirtiendo a formato compatible...


✅ Video mostrado arriba
